# Изненадани ли сте?

## Задача

Дадени са черно-бели изображения с размер 48х4на човешки лица, класифицирани в 7 категории според емоцията, на човека на съответната снимка: ядосан (angry), отвратен (disgusted), уплашен (fearful), щастлив (happy), неутрален (neutral), тъжен (sad) и изненадан (surprised). 

Някои примери от тренировъчните данни: 
![](https://www.googleapis.com/download/storage/v1/b/kaggle-user-content/o/inbox%2F8103790%2F3111a7794642911d765952df7c12659a%2Fexamples.png?generation=1781071421431342&alt=media)

Вашата задача е да съставите алгоритъм, който да може да разпознава тези емоции. 

Позволени са: 
- невронни мрежи
- класически алгоритми за машинно обучение
- техники за разширяване на данните (data augmentation)
- ансамблови методи

Забранени са:
- ръчно означаване на тестови изображения
- използване на тестови етикети
- търсене на тестовите изображения в интернет
- модели, които изпращат изображения към външни услуги
- използване на допълнителни набори от данни с означени емоции 
- предварително обучени модели 




### Метрика за оценка

Метриката за автоматична оценка е **F1 Macro**.  Тя изчислява средноаритметичната стойност на F1 резултата за всеки клас поотделно, третирайки всички класове като напълно равни, независимо от техния размер или брой.


### Submission файл 
Файлът за предаване има две колони - `image_id` и `label`. Примерна структура: 
```
image_id,label
028710.png,sad
028711.png,sad
028712.png,sad
028713.png,sad
028714.png,sad
```

# Базово решение

##  Импортиране на библиотеки

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import time
import joblib
import os
import zipfile

##  Зареждане на данните

In [ ]:
def decompress_zip(file_path, extract_to):
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

Signature: os.makedirs(name, mode=511, exist_ok=False)
Source:   
def makedirs(name, mode=0o777, exist_ok=False):
    """makedirs(name [, mode=0o777][, exist_ok=False])

    Super-mkdir; create a leaf directory and all intermediate ones.  Works like
    mkdir, except that any intermediate path segment (not just the rightmost)
    will be created if it does not exist. If the target directory already
    exists, raise an OSError if exist_ok is False. Otherwise no exception is
    raised.  This is recursive.

    """
    head, tail = path.split(name)
    if not tail:
        head, tail = path.split(head)
    if head and tail and not path.exists(head):
        try:
            makedirs(head, exist_ok=exist_ok)
        except FileExistsError:
            # Defeats race condition when another thread created the path
            pass
        cdir = curdir
        if isinstance(tail, bytes):
            cdir = bytes(curdir, 'ASCII')
        if tail == cdir:           # xxx/newdir/. exists if x

In [2]:
# Зареждане на подготвените данни
data = np.load('../data/processed/fer2013_processed.npz', allow_pickle=True)

X_train = data['X_train']
X_val = data['X_val']
X_test = data['X_test']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']
class_weights_array = data['class_weights']
EMOTIONS = list(data['emotions'])

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nЕмоции: {EMOTIONS}")

X_train shape: (6400, 48, 48)
X_val shape: (1600, 48, 48)
X_test shape: (7178, 48, 48)

Емоции: [np.str_('angry'), np.str_('disgusted'), np.str_('fearful'), np.str_('happy'), np.str_('neutral'), np.str_('sad'), np.str_('surprised')]


## Flatten на изображенията

Модели като Random Forest очакват 1D вектор като вход, не 2D изображение.

**Преобразуване:** `(N, 48, 48)` → `(N, 2304)`

Всеки пиксел става отделен feature (48 × 48 = 2304 features).

In [3]:
# Flatten: (N, 48, 48) -> (N, 2304)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print(f"X_train: {X_train.shape} -> {X_train_flat.shape}")
print(f"X_val: {X_val.shape} -> {X_val_flat.shape}")
print(f"X_test: {X_test.shape} -> {X_test_flat.shape}")
print(f"\nВсяко изображение е вектор с {X_train_flat.shape[1]} features")

X_train: (6400, 48, 48) -> (6400, 2304)
X_val: (1600, 48, 48) -> (1600, 2304)
X_test: (7178, 48, 48) -> (7178, 2304)

Всяко изображение е вектор с 2304 features


## 4. Обучение на Random Forest модел

In [5]:
print("Обучение на Random Forest модел...")
print()

start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train_flat, y_train)

rf_train_time = time.time() - start_time
print(f"\nВреме за обучение: {rf_train_time:.2f} секунди")

Обучение на Random Forest модел...



[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    1.5s



Време за обучение: 4.39 секунди


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    4.2s finished


In [6]:
# Предсказания на validation set
print("Предсказване на validation set...")
y_val_pred_rf = rf_model.predict(X_val_flat)

# Метрики
rf_val_accuracy = accuracy_score(y_val, y_val_pred_rf)
rf_val_f1 = f1_score(y_val, y_val_pred_rf, average='weighted')

print(f"\nRandom Forest Validation Резултати:")
print(f"  Accuracy: {rf_val_accuracy:.4f} ({rf_val_accuracy*100:.2f}%)")
print(f"  F1-Score (weighted): {rf_val_f1:.4f}")

Предсказване на validation set...

Random Forest Validation Резултати:
  Accuracy: 0.3881 (38.81%)
  F1-Score (weighted): 0.3556


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


# Задача 

Подобрете даденият базов модел или имплементирайте друг модел. 
Не се разрешава ползването на предварително обучен модел. 

In [ ]:
# Напишете своето решение тук 

## Финална оценка на Test Set

Test set използваме само веднъж - за финална оценка!

In [19]:
print(f"Финална оценка на {best_model_name} върху TEST set:")
print("=" * 50)

# Предсказания на test set
y_test_pred = best_model.predict(X_test_flat)

# Метрики
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test F1-Score (weighted): {test_f1:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=EMOTIONS))

Финална оценка на SVM върху TEST set:
Test Accuracy: 0.4289 (42.89%)
Test F1-Score (weighted): 0.4278

Classification Report:
              precision    recall  f1-score   support

       angry       0.31      0.29      0.30       958
     disgust       0.40      0.46      0.43       111
        fear       0.31      0.27      0.29      1024
       happy       0.59      0.55      0.57      1774
     neutral       0.40      0.42      0.41      1233
         sad       0.35      0.38      0.37      1247
    surprise       0.54      0.61      0.57       831

    accuracy                           0.43      7178
   macro avg       0.41      0.43      0.42      7178
weighted avg       0.43      0.43      0.43      7178



## 10. Запазване на модела

In [ ]:
# Създаване на директория за модели
os.makedirs('../models', exist_ok=True)

# Запазване на двата модела
joblib.dump(rf_model, '../models/rf_baseline.joblib')

print("Моделите са запазени:")

Моделите са запазени:
  - ../models/svm_baseline.joblib
  - ../models/rf_baseline.joblib
